## Adding UHGV confident, complete species reps to CheckV

In [ ]:
%%bash
### Download UHGV vOTU metadata and HQ+ genomes
wget https://portal.nersc.gov/UHGV/genome_catalogs/votus_hq_plus.fna.gz
wget https://portal.nersc.gov/UHGV/metadata/votus_metadata.tsv

In [ ]:
### Load metadta
import polars as pl

votus_metadata = pl.read_csv("votus_metadata.tsv", separator="\t", null_values="NA", ignore_errors=True)
votus_metadata.head(2)

uhgv_genome,uhgv_votu,checkv_completeness,checkv_quality,viral_confidence,genome_length,cds_count,ictv_taxonomy,uhgv_taxonomy,host_lineage,lifestyle,hyperprevalent,http_url
str,str,f64,str,str,i64,i64,str,str,str,str,str,str
"""UHGV-0121692""","""vOTU-000001""",100.0,"""Complete""","""Confident""",100403,93,"""root;Duplodnaviria;Heunggongvi…","""vFAM-00055;vSUBFAM-00155;vGENU…","""d__Bacteria;p__Firmicutes""","""lytic""","""Yes""","""https://portal.nersc.gov/UHGV/…"
"""UHGV-0169522""","""vOTU-000002""",100.0,"""Complete""","""Confident""",89551,122,"""root;Duplodnaviria;Heunggongvi…","""vFAM-00076;vSUBFAM-00080;vGENU…","""d__Bacteria;p__Bacteroidota;c_…","""lytic""","""Yes""","""https://portal.nersc.gov/UHGV/…"


In [ ]:
### Filter metadata to only complete, confident vOTU representatives
complete_votu_reps = (
    votus_metadata
        .filter(pl.col('checkv_quality') == 'Complete')
        .filter(pl.col('viral_confidence') == 'Confident')
)
print(f"Number of complete vOTU representatives: {complete_votu_reps.shape[0]}")

complete_votu_reps[['uhgv_genome']].write_csv("complete_votu_reps.csv", include_header=False)

Number of complete vOTU representatives: 24157


In [ ]:
%%bash
### Extract complete confident vOTU representative genomes
seqkit grep \
    votus_hq_plus.fna.gz \
    --pattern-file complete_votu_reps.csv \
    --out-file uhgv_hc_complete_votu_reps.fna.gz

# Check number of genomes in final file
zgrep -c "^>" uhgv_hc_complete_votu_reps.fna.gz
# 24157

[INFO] 24157 patterns loaded from file


24157


In [ ]:
%%bash
### Count number of genomes in CheckV database
zgrep -c "^>" /gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/UHVDB/toolkit/databases/checkv/1.0.3/checkv_db/checkv-db-v1.5/genome_db/checkv_reps.fna
# 62895

62895


In [ ]:
%%bash
### Identify UHGV confident, complete genomes not in CheckV representative database

# Create kmer-db ref
echo "/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/UHVDB/toolkit/databases/checkv/1.0.3/checkv_db/checkv-db-v1.5/genome_db/checkv_reps.fna" > ref_kdb.txt

# Build ref DB
kmer-db \
    build \
    -k 25 \
    -f 0.2 \
    -t 4 \
    -multisample-fasta \
    ref_kdb.txt \
    ref.kdb

# Create query db
echo "/mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb-manuscript/figure_s0/uhgv_hc_complete_votu_reps.fna.gz" > query_kdb.txt

# Compare query to ref
kmer-db \
    new2all \
    -sparse \
    -min num-kmers:20 \
    -min ani-shorter:0.95 \
    -t 4 \
    -multisample-fasta \
    ref.kdb \
    query_kdb.txt \
    query_v_ref.csv

# Convert output to distances
kmer-db \
    distance \
    ani-shorter \
    -sparse \
    -min 0.95 \
    -t 4 \
    query_v_ref.csv \
    query_v_ref.dist.csv

# Convert to LZ-ANI input format
/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/UHVDB/toolkit/bin/kmerdb_new2all_to_lzani.py \
    -i query_v_ref.dist.csv \
    -o query_v_ref.dist_mod.csv

# Create combined input file
cat /gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/UHVDB/toolkit/databases/checkv/1.0.3/checkv_db/checkv-db-v1.5/genome_db/checkv_reps.fna \
    > ref_query.combined.fna
zcat /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb-manuscript/figure_s0/uhgv_hc_complete_votu_reps.fna.gz \
    >> ref_query.combined.fna

# Align with LZ-ANI
lz-ani \
    all2all \
    --in-fasta ref_query.combined.fna \
    -o uhgv_hc_complete_v_checkv.lzani.tsv \
    --out-format query,reference,ani,qcov,rcov \
    -t 4 \
    --multisample-fasta true \
    --out-type tsv \
    --flt-kmerdb query_v_ref.dist_mod.csv 0.95

# Extract UHGV confident, complete genomes not in CheckV
csvtk filter2 \
    uhgv_hc_complete_v_checkv.lzani.tsv  \
    --tabs \
    --filter '( $ani >= 0.95 ) && ( $qcov >= 0.85 || $rcov >= 0.85 )' | \
csvtk cut \
    --tabs \
    --fields query | \
csvtk uniq \
    --tabs \
    --out-file uhgv_hc_complete_v_checkv.matches.tsv

seqkit grep \
    /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb-manuscript/figure_s0/uhgv_hc_complete_votu_reps.fna.gz \
    --threads 4 \
    --invert-match \
    --pattern-file uhgv_hc_complete_v_checkv.matches.tsv \
    --out-file uhgv_hc_complete_v_checkv.novel.fna.gz

# Count number of novel genomes
zgrep -c "^>" uhgv_hc_complete_v_checkv.novel.fna.gz
# 13565

In [ ]:
%%bash
### Run CheckV update
gunzip uhgv_hc_complete_v_checkv.novel.fna.gz

### Update CheckV database
checkv \
    update_database \
    /gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/UHVDB/toolkit/databases/checkv/1.0.3/checkv_db/checkv-db-v1.5 \
    /gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/UHVDB/toolkit/databases/checkv/uhgv_1 \
    uhgv_hc_complete_v_checkv.novel.fna \
    --threads 4

In [ ]:
%%bash
### Count size of updated databse
zgrep -c "^>" \
    /gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/UHVDB/toolkit/databases/checkv/uhgv_1/genome_db/checkv_reps.fna
# 76460